<a href="https://colab.research.google.com/github/Shahid-Ahmed-S/GenAI/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U langchain langchain-core langchain-groq langchain-community langchain-chroma langchain-huggingface langchain-text-splitters
!pip install -q chromadb sentence-transformers gradio

In [14]:
from google.colab import userdata
try:
    GROQ_API_KEY = userdata.get('gen-ai')
except:
    GROQ_API_KEY = "your-key"
print("Groq API key loaded successfully.")

Groq API key loaded successfully.


In [3]:
from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

print("Uploaded file:", file_name)

Saving from_mac.txt to from_mac.txt
Uploaded file: from_mac.txt


In [4]:
import os
os.listdir()

['.config', '.ipynb_checkpoints', 'from_mac.txt', 'sample_data']

In [5]:
with open("from_mac.txt", "r", encoding="utf-8") as file:
    text_content = file.read()

print("File loaded successfully.")
print("Total characters:", len(text_content))
print("\nPreview:")
print(text_content[:1000])

File loaded successfully.
Total characters: 2160

Preview:
Artificial Intelligence (AI) is the branch of computer science dedicated to creating systems that mimic human cognitive functions, such as learning, reasoning, problem-solving, and decision-making. Instead of following rigid, pre-written code, modern AI utilizes massive datasets and complex mathematical algorithms to identify intricate patterns, adapt to new information, and perform tasks that once strictly required human intellect. At the core of this technological evolution is Machine Learning, a method where computers analyze vast amounts of data to improve their performance over time without explicit programming. A more advanced subset, Deep Learning, uses artificial neural networks inspired by the human brain to process complex, unstructured data like audio, video, and natural language. Today, AI is firmly embedded in our daily lives, driving technologies like autonomous vehicles, facial recognition, personalized medical d

In [6]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("from_mac.txt", encoding="utf-8")

documents = loader.load()

print("Number of documents loaded:", len(documents))
print("\nDocument preview:")
print(documents[0].page_content[:1000])

/tmp/ipykernel_17648/1265982381.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Number of documents loaded: 1

Document preview:
Artificial Intelligence (AI) is the branch of computer science dedicated to creating systems that mimic human cognitive functions, such as learning, reasoning, problem-solving, and decision-making. Instead of following rigid, pre-written code, modern AI utilizes massive datasets and complex mathematical algorithms to identify intricate patterns, adapt to new information, and perform tasks that once strictly required human intellect. At the core of this technological evolution is Machine Learning, a method where computers analyze vast amounts of data to improve their performance over time without explicit programming. A more advanced subset, Deep Learning, uses artificial neural networks inspired by the human brain to process complex, unstructured data like audio, video, and natural language. Today, AI is firmly embedded in our daily lives, driving technologies like autonomous vehicles, facial recognition, personalized medical diagnostics

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Number of chunks created:", len(chunks))

for i, chunk in enumerate(chunks[:5]):
    print("\nChunk", i + 1)
    print(chunk.page_content)
    print("-" * 50)

Number of chunks created: 9

Chunk 1
Artificial Intelligence (AI) is the branch of computer science dedicated to creating systems that mimic human cognitive functions, such as learning, reasoning, problem-solving, and decision-making. Instead of following rigid, pre-written code, modern AI utilizes massive datasets and complex
--------------------------------------------------

Chunk 2
modern AI utilizes massive datasets and complex mathematical algorithms to identify intricate patterns, adapt to new information, and perform tasks that once strictly required human intellect. At the core of this technological evolution is Machine Learning, a method where computers analyze vast
--------------------------------------------------

Chunk 3
Learning, a method where computers analyze vast amounts of data to improve their performance over time without explicit programming. A more advanced subset, Deep Learning, uses artificial neural networks inspired by the human brain to process complex, uns

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings model loaded successfully.


In [9]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="student_rag_collection"
)

print("Vector store created successfully.")

Vector store created successfully.


In [35]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Create LLM using a valid Groq model name
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="openai/gpt-oss-20b"
)

# Create retriever from vector store
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

# Function to join retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create RAG prompt
rag_prompt = ChatPromptTemplate.from_template(
    '''
Answer the question using only the context below.
If the answer is not available in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:
'''
)

# Create Runnable RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("Runnable RAG chain created successfully with valid model.")

Runnable RAG chain created successfully with valid model.


In [36]:
question = "How do I create a variable in Python?"

answer = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:")
print(answer)

Question: How do I create a variable in Python?

Answer:
I don't know.


In [37]:
question = "What are functions?"

retrieved_docs = retriever.invoke(question)

print("Question:", question)
print("\nRetrieved chunks:")

for i, doc in enumerate(retrieved_docs):
    print("\nChunk", i + 1)
    print(doc.page_content)
    print("-" * 60)

Question: What are functions?

Retrieved chunks:

Chunk 1
Learning, a method where computers analyze vast amounts of data to improve their performance over time without explicit programming. A more advanced subset, Deep Learning, uses artificial neural networks inspired by the human brain to process complex, unstructured data like audio, video, and
------------------------------------------------------------

Chunk 2
automation and data privacy. Ultimately, artificial intelligence is reshaping the global economy, acting as a powerful tool that enhances human capability while requiring careful ethical oversight to ensure its safe integration into society.
------------------------------------------------------------

Chunk 3
Artificial Intelligence (AI) is the branch of computer science dedicated to creating systems that mimic human cognitive functions, such as learning, reasoning, problem-solving, and decision-making. Instead of following rigid, pre-written code, modern AI utilizes mass

In [38]:
questions = [
    "What is Artificial Intelligence?",
    "Where is this file created?",
    "Where will the file be transfered?",
    "How is the file transfered?"
]

for q in questions:
    print("\nQuestion:", q)
    print("Answer:", rag_chain.invoke(q))
    print("-" * 70)


Question: What is Artificial Intelligence?
Answer: Artificial Intelligence (AI) is the branch of computer science dedicated to creating systems that mimic human cognitive functions, such as learning, reasoning, problem‑solving, and decision‑making. Instead of following rigid, pre‑written code, modern AI utilizes massive datasets and complex mathematical algorithms to identify intricate patterns, adapt to new information, and perform tasks that once strictly required human intellect.
----------------------------------------------------------------------

Question: Where is this file created?
Answer: Mac.
----------------------------------------------------------------------

Question: Where will the file be transfered?
Answer: The file will be transferred to the Windows Laptop.
----------------------------------------------------------------------

Question: How is the file transfered?
Answer: It’s transferred with SCP (Secure Copy), which uses an SSH connection between the Mac and the

In [39]:
def ask_rag(question):
    if question.strip() == "":
        return "Please enter a question."

    return rag_chain.invoke(question)

print(ask_rag("What are variables?"))

I don't know.


In [40]:
import gradio as gr

def answer_question(question):
    return ask_rag(question)

app = gr.Interface(
    fn=answer_question,

    inputs=gr.Textbox(
        label="Your Question",
        lines=3,
        placeholder="Ask a question from your uploaded document..."
    ),

    outputs=gr.Textbox(
        label="Answer",
        lines=10,
        max_lines=25
    ),

    title="RAG Q&A App using Runnables",
    description="Upload a text document, build a runnable RAG chain, and ask questions from the document.",

    examples=[
        "What are variables?",
        "How do I use loops?",
        "Explain functions",
        "What is a list?"
    ]
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b7d67485c5fad6996e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
